<a href="https://colab.research.google.com/github/khushikadway143-star/bts-scientific-calculator/blob/main/Prescription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# 1. Paste your copied token inside the quotes below
KAGGLE_TOKEN = "KGAT_a9123cb840e56debf69cee50102c3cd0"

# 2. Set environment variables
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

# 3. Install Kaggle CLI
!pip install -q kaggle

In [2]:
import kagglehub

# Download latest version / Doctor Handwriting Dataset
path1 = kagglehub.dataset_download("mrdude20/doctor-handwriting-recognition-dataset")

print("Path to dataset files:", path1)

100%|██████████| 7.28M/7.28M [00:00<00:00, 51.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mrdude20/doctor-handwriting-recognition-dataset/versions/1


In [3]:
# Download latest version / OCR Dataset
path2 = kagglehub.dataset_download("msharathgowda/ocr-dataset")

print("Path to dataset files:", path2)

100%|██████████| 1.49G/1.49G [00:16<00:00, 94.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/msharathgowda/ocr-dataset/versions/1


In [4]:
# Inspect the contents of the downloaded folder
print("Folders/Files in path1:: ")
for root, dirs, files in os.walk(path1):
    print("Folder Path:", root)
    print("Subfolders:", dirs)
    print("Sample Files:", files[:5])  # Show up to 5 sample files
    print("-" * 40)

Folders/Files in path1:: 
Folder Path: /root/.cache/kagglehub/datasets/mrdude20/doctor-handwriting-recognition-dataset/versions/1
Subfolders: ['img']
Sample Files: ['doctor_handwriting_labels.csv']
----------------------------------------
Folder Path: /root/.cache/kagglehub/datasets/mrdude20/doctor-handwriting-recognition-dataset/versions/1/img
Subfolders: ['img']
Sample Files: []
----------------------------------------
Folder Path: /root/.cache/kagglehub/datasets/mrdude20/doctor-handwriting-recognition-dataset/versions/1/img/img
Subfolders: []
Sample Files: ['img 15.jpg', 'IMG_20241006_144408 - Copy.jpg', 'IMG_20241006_143132 - Copy.jpg', 'IMG_20241006_153240.jpg', 'IMG_20241006_152156.jpg']
----------------------------------------


In [5]:
import pandas as pd

In [6]:
# Define exact paths based on your inspection
csv_path = os.path.join(path1, "doctor_handwriting_labels.csv")
img_dir = os.path.join(path1, "img", "img")

In [7]:
# Load the CSV file
df = pd.read_csv(csv_path)

In [8]:
# Display dataset details
print("Total rows in CSV:", len(df))

Total rows in CSV: 89


In [9]:
print("\nFirst 5 rows of the dataset:")
print(df.head())


First 5 rows of the dataset:
                                            filename         label
0  .trashed-1730802496-IMG_20241006_151957 - Copy...  cefiget 40mg
1                     IMG_20241006_141949 - Copy.jpg    tab azimax
2                     IMG_20241006_142328 - Copy.jpg  tab toniflex
3                     IMG_20241006_142533 - Copy.jpg          nims
4                     IMG_20241006_142757 - Copy.jpg      dicloran


In [10]:
print("\n Last 5 rows of the dataset:")
print(df.tail())


 Last 5 rows of the dataset:
     filename    label
84  img 5.jpg   breeky
85  img 6.jpg   provas
86  img 7.jpg    movax
87  img 8.jpg  caricef
88  img 9.jpg  bisleri


In [11]:
import cv2
import numpy as np
from PIL import Image

In [12]:
def preprocess_prescription_image(image_input):
    """
    Cleans background shadows and normalizes image for TrOCR
    """
    # Convert PIL Image or path to OpenCV BGR array
    if isinstance(image_input, str):
        img = cv2.imread(image_input)
    else:
        img = cv2.cvtColor(np.array(image_input), cv2.COLOR_RGB2BGR)

    # 1. Convert to Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Adaptive Gaussian Thresholding (Removes lighting/shadow variations)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )

    # 3. Convert back to 3-Channel RGB PIL Image for TrOCR Processor
    cleaned_rgb = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(cleaned_rgb)

# Test cleaning pipeline on a sample image from dataset
sample_img_path = os.path.join(path1, "img", "img", df.iloc[0]['filename'])
if os.path.exists(sample_img_path):
    cleaned_sample = preprocess_prescription_image(sample_img_path)
    print("OpenCV Image Preprocessing Successful! Output dimensions:", cleaned_sample.size)

OpenCV Image Preprocessing Successful! Output dimensions: (1335, 447)


CELL 3: Load Microsoft TrOCR Model & Processor

In [13]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [15]:
# Determine GPU device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running execution pipeline on: {device}")

Running execution pipeline on: cuda


In [16]:
# Load pre-trained Microsoft TrOCR processor and model
MODEL_NAME = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

def predict_handwritten_text(image_input):
    """
    Processes raw prescription image and returns predicted text
    """
    cleaned_img = preprocess_prescription_image(image_input)

    # Convert image pixels to tensor (384x384 standard grid)
    pixel_values = processor(images=cleaned_img, return_tensors="pt").pixel_values.to(device)

    # Autoregressive generation using Vision Transformer + Language Decoder
    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=64)

    extracted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return extracted_text

print("TrOCR Pipeline Loaded & Ready!")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TrOCR Pipeline Loaded & Ready!


STEP 1: TRAIN / VALIDATION / TEST SPLIT (80% - 10% - 10%)

In [27]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [28]:
csv_path = os.path.join(path1, "doctor_handwriting_labels.csv")
df = pd.read_csv(csv_path)

In [29]:
# Drop any rows with missing or blank labels
df = df.dropna(subset=['label']).reset_index(drop=True)

In [30]:
# 1. First Split: 80% Train, 20% Temp (Val + Test)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)

In [31]:
# 2. Second Split: Split the 20% into 10% Validation and 10% Testing
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

In [32]:
print(f"Total Dataset Count : {len(df)}")

Total Dataset Count : 89


In [33]:
print(f"Training Set (80%)  : {len(train_df)} samples")

Training Set (80%)  : 71 samples


In [34]:
print(f"Validation Set (10%): {len(val_df)} samples")

Validation Set (10%): 9 samples


In [35]:
print(f"Testing Set (10%)   : {len(test_df)} samples")

Testing Set (10%)   : 9 samples


STEP 2 & 3: PYTORCH DATASET LOADER & TOKENIZATION

In [36]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [37]:
class PrescriptionPyTorchDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_target_length=128):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_name = row['filename']
        text_label = str(row['label'])

In [40]:
# Load image file
csv_path = os.path.join(path1, "doctor_handwriting_labels.csv")

In [43]:
# 3. Get the filename of the first image from the DataFrame
first_image_filename = df.iloc[0]['filename']

In [44]:
# 4. Correct path to the image file inside the img/img subfolder
image_path = os.path.join(path1, "img", "img", first_image_filename)

In [45]:
# 5. Open the actual image file
image = Image.open(image_path).convert("RGB")
print("Image successfully loaded! Dimensions:", image.size)

Image successfully loaded! Dimensions: (1335, 447)


In [46]:
# Apply OpenCV preprocessing routine
cleaned_image = preprocess_prescription_image(image)

In [48]:
# Encode Image into Pixel Tensors (Shape: 3, 384, 384)
pixel_values = processor(cleaned_image, return_tensors="pt").pixel_values.squeeze(0)

In [50]:
print("Pixel Tensor successfully created!")
print("Tensor Shape:", pixel_values.shape)

Pixel Tensor successfully created!
Tensor Shape: torch.Size([3, 384, 384])


In [52]:
# 1. Define sample text label and max length for testing
text_label = str(df.iloc[0]['label'])
max_target_length = 128

# 2. Encode Text Label into Token IDs using TrOCR Tokenizer
labels = processor.tokenizer(
    text_label,
    padding="max_length",
    max_length=max_target_length,
    truncation=True
).input_ids

print("Text Tokenization Successful!")
print("Sample Text Label:", text_label)
print("First 10 Token IDs:", labels[:10])

Text Tokenization Successful!
Sample Text Label: cefiget 40mg
First 10 Token IDs: [0, 1755, 22699, 594, 843, 22984, 2, 1, 1, 1]


In [54]:
# Replace padding token ID (-100) so PyTorch Cross-Entropy Loss ignores padding
# 1. Replace padding token ID (-100) so PyTorch Cross-Entropy Loss ignores padding
labels = [label if label != processor.tokenizer.pad_token_id else -100 for label in labels]

# 2. Store output in a dictionary (instead of using 'return')
sample_batch = {
    "pixel_values": pixel_values,
    "labels": torch.tensor(labels)
}

print("Batch Dictionary Created Successfully!")
print("Pixel Values Shape:", sample_batch["pixel_values"].shape)
print("Labels Tensor Shape:", sample_batch["labels"].shape)

Batch Dictionary Created Successfully!
Pixel Values Shape: torch.Size([3, 384, 384])
Labels Tensor Shape: torch.Size([128])


In [55]:
# Instantiate PyTorch Datasets
img_directory = os.path.join(path1, "img", "img")
train_dataset = PrescriptionPyTorchDataset(train_df, img_directory, processor)
val_dataset   = PrescriptionPyTorchDataset(val_df, img_directory, processor)
test_dataset  = PrescriptionPyTorchDataset(test_df, img_directory, processor)

In [56]:
# Create DataLoaders for batch processing
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

print("PyTorch DataLoaders Successfully Initialized!")

PyTorch DataLoaders Successfully Initialized!


STEP 4 & 5: ACCURACY EVALUATION (CER & WER METRICS)

In [58]:
!pip install -q evaluate jiwer

import torch
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.6 MB/s eta 0:00:00


In [59]:
# 2. Load HuggingFace official metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

In [65]:
def evaluate_model_performance(model, data_loader, processor, device):
    """
    Calculates Character Error Rate (CER) and Word Error Rate (WER) on Test Set
    """
    model.eval()
    predictions_list = []
    references_list = []

    print("Evaluating Model Accuracy on Test Set...")

    with torch.no_grad():
      for batch in data_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # Generate token IDs from model
            outputs = model.generate(pixel_values, max_new_tokens=64)

            # Decode predicted token IDs into text strings
            pred_texts = processor.batch_decode(outputs, skip_special_tokens=True)

            # Replace -100 in labels back to pad token ID for decoding
            labels[labels == -100] = processor.tokenizer.pad_token_id
            ref_texts = processor.batch_decode(labels, skip_special_tokens=True)

            predictions_list.extend(pred_texts)
            references_list.extend(ref_texts)

    # 2. Calculate metrics (outside loop, properly indented)
    cer_score = cer_metric.compute(predictions=predictions_list, references=references_list)
    wer_score = wer_metric.compute(predictions=predictions_list, references=references_list)

    # 3. Corrected accuracy calculation formula
    accuracy_percentage = max(0.0, (1.0 - cer_score) * 100)

    print("\n================== MODEL PERFORMANCE METRICS ==================")
    print(f" Character Error Rate (CER) : {cer_score * 100:.2f}% (Lower is better)")
    print(f" Word Error Rate (WER)      : {wer_score * 100:.2f}% (Lower is better)")
    print(f" Estimated Model Accuracy   : {accuracy_percentage:.2f}%")
    print("===============================================================\n")

    return predictions_list, references_list

# 4. Run evaluation on test dataset loader
sample_preds, sample_refs = evaluate_model_performance(model, test_loader, processor, device)

Evaluating Model Accuracy on Test Set...


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'NoneType'>

CELL 4: Expose FastAPI Inference Server for Mobile App via ngrok

In [19]:
!pip install -q pyngrok fastapi uvicorn nest_asyncio

In [20]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile
from pyngrok import ngrok
import io

In [21]:
app = FastAPI(title="Prescription Reader API")

In [22]:
@app.get("/")
def home():
    return {"status": "Prescription OCR API is Active"}

In [23]:
@app.post("/predict")
async def predict_prescription(file: UploadFile = File(...)):
    # Read uploaded file bytes from mobile camera request
    contents = await file.read()
    raw_image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Run TrOCR prediction
    extracted_text = predict_handwritten_text(raw_image)

    return {
        "status": "success",
        "raw_text": extracted_text,
        "parsed_entities": {
            "medicine_name": extracted_text,
            "verification_status": "pending_human_review"
        }
    }

In [24]:
# Apply nest_asyncio to allow running Uvicorn inside Colab
nest_asyncio.apply()

In [25]:
# Enter your free ngrok token below (Get yours at https://dashboard.ngrok.com)
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [26]:
# Open public HTTP tunnel to port 8000
public_url = ngrok.connect(8000)
print(f" MOBILE API ENDPOINT URL ")
print(f"Use this URL in your Mobile App: {public_url.public_url}/predict")
print(f"----------")

ERROR:pyngrok.process.ngrok:t=2026-10-13T18:01:59+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTHTOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-13T18:01:59+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTHTOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-10-13T18:01:59+0000 lvl=crit msg="command failed" err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoke

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_AUTHTOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.